# 01 — Semantic metadata analysis (RQ1–RQ3)
Consumes only the approved frozen master JSONL. Produces machine-readable semantic coverage, overlap, completeness, and richness results.


In [ ]:
from pathlib import Path
import json, subprocess, sys, csv
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
REPO=Path('/content/research_software_classification_attributes')
if not REPO.exists():
    subprocess.run(['git','clone','-b','data-finalization','https://github.com/kuefmz/research_software_classification_attributes.git',str(REPO)],check=True)
sys.path.insert(0,str(REPO))
ROOT=Path('/content/drive/MyDrive/phd_research_software')
FROZEN=ROOT/'data/frozen'; OUT=ROOT/'results/semantic_metadata'; OUT.mkdir(parents=True,exist_ok=True)


In [ ]:
from src.experiments.io import load_manifest
from src.experiments.semantic import semantic_analysis
manifest=load_manifest(FROZEN/'MANIFEST.json')
result=semantic_analysis(FROZEN/'master_dataset.jsonl')
result['dataset_release_identifier']=manifest['dataset_release_identifier']
result['dataset_hash']=next(x['sha256'] for x in manifest['outputs'] if x['path'].endswith('master_dataset.jsonl'))
(OUT/'rq1_rq2_rq3_semantic_analysis.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result,indent=2)[:12000])


In [ ]:
with (OUT/'field_coverage.csv').open('w',newline='') as f:
    w=csv.writer(f); w.writerow(['source','field','coverage'])
    for source,vals in result['field_coverage'].items():
        for field,value in vals.items(): w.writerow([source,field,value])
with (OUT/'dimension_coverage.csv').open('w',newline='') as f:
    w=csv.writer(f); w.writerow(['source','dimension','coverage'])
    for source,vals in result['dimension_coverage'].items():
        for dim,value in vals.items(): w.writerow([source,dim,value])
with (OUT/'overlap.csv').open('w',newline='') as f:
    w=csv.writer(f); w.writerow(['measure','value'])
    for k,v in result['overlap'].items(): w.writerow([k,json.dumps(v) if isinstance(v,dict) else v])
